In [14]:
# ═════════════════════════════════════════════════════════════════
#  Wildfires Map - Active Fire Hotspots
#  Author: Germán Alberto Giménez Silva <ggerman@gmail.com>
#  Library: libgd-gis / ruby-libgd
#  Description: Active fires detected by MODIS satellites (last 24h).
#               Orange markers indicate thermal intensity.
#               Data: NASA FIRMS (firms.modaps.eosdis.nasa.gov)
# ═════════════════════════════════════════════════════════════════

require 'gd'
require 'gd/gis'
require 'net/http'

def render(file_name)
    IRuby.display "<img src='#{file_name}' />", mime: "text/html"
end

def polaroid_frame(image, margin: 20, bottom_margin: 80, bg_color: [255, 255, 255, 255])
  new_width  = image.width + margin * 2
  new_height = image.height + margin + bottom_margin

  framed = GD::Image.new(new_width, new_height)
  framed.antialias = false
  bg = GD::Color.rgba(*bg_color)
  framed.filled_rectangle(0, 0, new_width, new_height, bg)
  framed.copy(image, margin, margin, 0, 0, image.width, image.height)
  framed
end

puts "Fetching active fires from NASA FIRMS..."

url = "https://firms.modaps.eosdis.nasa.gov/active_fire/c6/text/MODIS_C6_global_24h.csv"

begin
  response = Net::HTTP.get(URI(url))
  lines = response.strip.split("\n")

  fires = []
  lines[1..-1].each do |line|
    parts = line.split(',')
    next if parts.size < 7

    lon = parts[0].to_f
    lat = parts[1].to_f
    brightness = parts[2].to_f

    next unless lat.between?(-85, 85) && lon.between?(-180, 180)

    fires << { lon: lon, lat: lat, brightness: brightness }
    break if fires.size >= 300
  end

  puts "Retrieved #{fires.size} fire hotspots"
rescue => e
  puts "Error: #{e.message}"
  fires = []
end

if fires.any?
  map = GD::GIS::Map.new(
    bbox: :africa,
    zoom: 5,
    basemap: :esri_satellite,
    width: 1000,
    height: 600
  )

  map.style = GD::GIS::Style.load("light")
  font_path = GD::GIS::FontHelper.find("DejaVuSans") || GD::GIS::FontHelper.random
  map.style.points[:font] ||= font_path
  map.style.points[:size] ||= 12

  # High intensity fires (hotter)
  high = fires.select { |f| f[:brightness] > 320 }
  if high.any?
    data = high.map { |f| { lon: f[:lon], lat: f[:lat], label: "*" } }
    layer = GD::GIS::PointsLayer.new(
      data, lon: ->(p) { p[:lon] }, lat: ->(p) { p[:lat] },
      icon: "numeric", label: ->(p) { p[:label] },
      font: font_path, size: 16,
      color: [255, 80, 40, 220],
      font_color: [255, 255, 255, 255]
    )
    map.instance_variable_get(:@points_layers) << layer
    puts "High intensity: #{high.size}"
  end

  # Low intensity fires
  low = fires - high
  if low.any?
    data = low.map { |f| { lon: f[:lon], lat: f[:lat], label: "*" } }
    layer = GD::GIS::PointsLayer.new(
      data, lon: ->(p) { p[:lon] }, lat: ->(p) { p[:lat] },
      icon: "numeric", label: ->(p) { p[:label] },
      font: font_path, size: 10,
      color: [255, 140, 60, 180],
      font_color: [255, 255, 255, 255]
    )
    map.instance_variable_get(:@points_layers) << layer
    puts "Low intensity: #{low.size}"
  end

  map.render

  framed = polaroid_frame(map.image, margin: 20, bottom_margin: 90)

  framed.text_ft(
    "Active Wildfires -- Last 24 Hours (Western Africa)",
    x: 30, y: framed.height - 70,
    font: font_path, size: 14,
    color: GD::Color.rgb(40, 40, 40)
  )

  framed.text_ft(
    "Orange markers = fire hotspots | Larger = higher intensity | Data: NASA FIRMS",
    x: 30, y: framed.height - 50,
    font: font_path, size: 11,
    color: GD::Color.rgb(80, 80, 80)
  )

  framed.text_ft(
    "German A. Gimenez Silva <ggerman@gmail.com> | libgd-gis / ruby-libgd",
    x: 30, y: framed.height - 30,
    font: font_path, size: 10,
    color: GD::Color.rgb(120, 120, 120)
  )

  framed.save("/work/wildfires_framed.png")
  render("wildfires_framed.png")
  puts "Saved: wildfires_framed.png"
else
  puts "No fire data available"
end

Fetching active fires from NASA FIRMS...
Retrieved 3 fire hotspots
Low intensity: 3


"<img src='wildfires_framed.png' />"

Saved: wildfires_framed.png
